## Cell 1 : Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import warnings
warnings.filterwarnings('ignore')

# Setup estetika visualisasi (Ukuran default, resolusi, dan warna)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style="whitegrid", palette="muted")

print("Library berhasil dimuat dan visualisasi sudah disetel!")

## Cell 2 : Koneksi dan Tarik Data 

In [ ]:
# Pastikan path ini sesuai dengan posisi .env milikmu
load_dotenv('../.env', override=True)

# Ambil URL MLflow utuh 
db_url_mlflow = os.getenv("DATABASE_URL_DIRECT")

if not db_url_mlflow:
    raise ValueError("DATABASE_URL_DIRECT tidak ditemukan di .env!")

# Buat URL polos khusus untuk Pandas & SQLAlchemy
db_url_pandas = db_url_mlflow.split("?")[0]

print("Membuka jembatan ke Supabase...")

try:
    engine = create_engine(db_url_pandas)
    
    # Menarik data historis sekaligus melakukan JOIN dengan nama wilayah
    query = """
        SELECT 
            dh.waktu_aktual,
            wd.nama_wilayah,
            dh.pm25, dh.pm10, dh.so2, dh.co, dh.no2, dh.ozon
        FROM data_historis dh
        JOIN wilayah_details wd ON dh.id_wilayah = wd.id_wilayah
        ORDER BY dh.waktu_aktual ASC
    """
    
    df = pd.read_sql(query, engine)
    
    # Pastikan format waktu sudah datetime
    df['waktu_aktual'] = pd.to_datetime(df['waktu_aktual'])
    
    print(f"Berhasil menarik {df.shape[0]} baris dan {df.shape[1]} kolom dari Supabase.")
    display(df.head(3))
    
except Exception as e:
    print(f"Gagal terhubung ke database: {e}")

## Cell 3 : Membedah Isi Kolom

In [ ]:
# 1. Menampilkan Tipe Data dan Missing Values
print("=== INFO DATASET ===")
df.info()

print("\n=== JUMLAH MISSING VALUES ===")
print(df.isna().sum())

print("\n=== DESKRIPSI FITUR (KOLOM) ===")
print("- waktu_aktual : Waktu pencatatan riil (timestamp) saat data ditarik dari sensor/OpenWeather.")
print("- nama_wilayah : Nama entitas geografis (Kota/Kabupaten) di wilayah Jawa Timur.")
print("- pm25, pm10   : Konsentrasi partikel debu halus dalam satuan microgram/m3 (µg/m³).")
print("- so2, no2     : Konsentrasi gas Sulfur Dioksida dan Nitrogen Dioksida (berkaitan dengan emisi industri/kendaraan).")
print("- co           : Konsentrasi gas Karbon Monoksida (parameter utama pembakaran tidak sempurna).")
print("- ozon         : Konsentrasi gas O3 permukaan (polutan sekunder dari reaksi fotokimia).")

# 2. Statistik Deskriptif Dasar
display(df.describe())

## Cell 4 : Distribusi Polutan

In [ ]:
daftar_polutan = ['pm25', 'pm10', 'so2', 'co', 'no2', 'ozon']

# Membuat grid 2 baris x 3 kolom
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, polutan in enumerate(daftar_polutan):
    sns.histplot(df[polutan], kde=True, ax=axes[i], color='teal', bins=30)
    
    # Pengaturan judul dan label yang presisi
    axes[i].set_title(f'Distribusi {polutan.upper()}', loc='center', pad=10, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Konsentrasi (µg/m³)', fontsize=10)
    axes[i].set_ylabel('Frekuensi', fontsize=10)

# Merapikan jarak antar grafik agar tidak bertumpuk
plt.tight_layout()
plt.show()

# print("Insight: Distribusi polutan cenderung 'Right-Skewed' (menumpuk di angka kecil dengan ekor panjang ke kanan).")
# print("Ini membenarkan keputusan kita melakukan Transformasi Logaritmik (Log1p) saat modeling agar data lebih stabil dipelajari AI.")

## Cell 5 : Korelasi Antar Polutan

In [ ]:
# Menghitung korelasi Pearson
korelasi = df[daftar_polutan].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(korelasi, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)

# Pengaturan judul dan perataan
plt.title('Matriks Korelasi Antar Polutan Udara', loc='center', pad=15, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# print("Insight: Jika terdapat polutan dengan korelasi positif yang cukup kuat (misal warna merah gelap),")
# print("itu membuktikan bahwa pendekatan pemodelan Multivariate XGBoost kita sudah sangat tepat dibandingkan regresi individu.")

## Cell 6 : Eksplorasi Pola Waktu

In [ ]:
# Mengekstrak fitur Jam
df['jam'] = df['waktu_aktual'].dt.hour

# Menghitung rata-rata polusi per jam
tren_harian = df.groupby('jam')[daftar_polutan].mean()

plt.figure(figsize=(12, 5))
sns.lineplot(data=tren_harian, dashes=False, palette='tab10', linewidth=2)

# Format visualisasi sumbu dan judul
plt.title('Rata-rata Konsentrasi Polutan Berdasarkan Jam (Dalam 24 Jam)', loc='center', pad=15, fontsize=14, fontweight='bold')
plt.xlabel('Jam (00:00 - 23:00)', fontsize=12)
plt.ylabel('Rata-rata Konsentrasi (µg/m³)', fontsize=12)
plt.xticks(range(0, 24))
plt.legend(title='Jenis Polutan', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

## Cell 7 : Eksplorasi Wilayah

In [ ]:
# Menghitung rata-rata PM2.5 per kota, ambil 10 teratas
top_kota = df.groupby('nama_wilayah')['pm25'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_kota.values, y=top_kota.index, palette='viridis')

# Format visualisasi yang rapi
plt.title('10 Wilayah dengan Rata-rata PM2.5 Tertinggi', loc='center', pad=15, fontsize=14, fontweight='bold')
plt.xlabel('Rata-rata PM2.5 (µg/m³)', fontsize=12)
plt.ylabel('Nama Wilayah', fontsize=12)

plt.tight_layout()
plt.show()

print("Insight: Perbedaan garis dasar (baseline) polusi antar kota ini yang mendasari penggunaan One-Hot Encoding pada id_wilayah.")